<a href="https://colab.research.google.com/github/VictorNevola/ml-study/blob/main/ml_09_naive_bayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mp
import os

OUT = "slides_aula02"                  # output folder
os.makedirs(OUT, exist_ok=True)        # create it if missing (fixes the FileNotFoundError)
BLUE, RED, GREEN, GREY = '#1f77b4', '#D62728', '#2CA02C', '#888888'

# ---------- SLIDE 1: Bayes intuition (belief + evidence) ----------
fig, ax = plt.subplots(figsize=(13, 6.2))
ax.axis("off")
ax.text(0.5, 0.95, "Bayes' intuition: start with a belief and UPDATE it with evidence",
        ha='center', fontsize=17, fontweight='bold', transform=ax.transAxes)

boxes = [
    (0.03, "INITIAL BELIEF", "Before reading anything:\n13% of messages\nare spam", '#DDDDDD', "13%"),
    (0.37, "+ EVIDENCE", "The message contains\nthe word\n'PRIZE'", '#FFE9A8', "->"),
    (0.71, "UPDATED BELIEF", "Now the chance of\nbeing spam is\nMUCH higher", '#FFB3B3', "97%"),
]
for x, title, body, color, big in boxes:
    ax.add_patch(mp.FancyBboxPatch((x, 0.28), 0.26, 0.5, boxstyle="round,pad=0.02",
                                   facecolor=color, edgecolor='black', lw=2,
                                   transform=ax.transAxes))
    ax.text(x+0.13, 0.72, title, ha='center', fontsize=13, fontweight='bold', transform=ax.transAxes)
    ax.text(x+0.13, 0.585, body, ha='center', fontsize=12, transform=ax.transAxes)
    ax.text(x+0.13, 0.36, big, ha='center', fontsize=26, fontweight='bold',
            color='#333', transform=ax.transAxes)

for x in (0.30, 0.64):
    ax.annotate("", xy=(x+0.06, 0.53), xytext=(x, 0.53), xycoords='axes fraction',
                arrowprops=dict(arrowstyle="-|>", lw=3, color='#333'))

ax.text(0.5, 0.13, "Naive Bayes does this with EVERY word in the message, one after another.",
        ha='center', fontsize=14, style='italic', color='#444', transform=ax.transAxes)
plt.savefig(os.path.join(OUT, "nb_intuicao_bayes.png"), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT, "nb_intuicao_bayes.svg"), bbox_inches='tight')
print("1 ok")

# ---------- SLIDE 2: Why "NAIVE" ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax in axes: ax.axis("off")

axes[0].text(0.5, 0.93, "What Naive Bayes ASSUMES", ha='center', fontsize=15,
             fontweight='bold', color=RED, transform=axes[0].transAxes)
axes[0].text(0.5, 0.80, '"The words are INDEPENDENT"', ha='center', fontsize=13,
             style='italic', transform=axes[0].transAxes)
words = ["FREE", "PRIZE", "CLAIM", "NOW"]
for i, w in enumerate(words):
    axes[0].add_patch(mp.FancyBboxPatch((0.08+i*0.22, 0.45), 0.18, 0.14,
                      boxstyle="round,pad=0.02", facecolor='#FFD9D9',
                      edgecolor=RED, lw=2, transform=axes[0].transAxes))
    axes[0].text(0.17+i*0.22, 0.52, w, ha='center', va='center', fontsize=12,
                 fontweight='bold', transform=axes[0].transAxes)
axes[0].text(0.5, 0.28, "Each word 'votes' on its own,\nas if word order and context\ndid not exist.",
             ha='center', fontsize=12, transform=axes[0].transAxes)
axes[0].text(0.5, 0.10, "X   This is FALSE in real life!", ha='center', fontsize=13,
             fontweight='bold', color=RED, transform=axes[0].transAxes)

axes[1].text(0.5, 0.93, "And yet... IT WORKS", ha='center', fontsize=15,
             fontweight='bold', color=GREEN, transform=axes[1].transAxes)
axes[1].text(0.5, 0.80, "98.7% accuracy on spam", ha='center', fontsize=13,
             style='italic', transform=axes[1].transAxes)
axes[1].text(0.5, 0.50,
             "To DECIDE between two classes,\nthe model does not need the\nexact probabilities --\n\nit only needs the correct class\nto get the HIGHEST score.",
             ha='center', fontsize=13, transform=axes[1].transAxes)
axes[1].text(0.5, 0.14, "Wrong assumption,\nRIGHT decision.", ha='center', fontsize=14,
             fontweight='bold', color=GREEN, transform=axes[1].transAxes)

fig.suptitle('Why "NAIVE"?', fontsize=19, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(OUT, "nb_por_que_naive.png"), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT, "nb_por_que_naive.svg"), bbox_inches='tight')
print("2 ok")

# ---------- SLIDE 3: Bag of Words ----------
fig, ax = plt.subplots(figsize=(13.5, 6))
ax.axis("off")
ax.text(0.5, 0.96, "How text becomes numbers: the Bag of Words",
        ha='center', fontsize=17, fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.85, '"Win a FREE prize now"', ha='center', fontsize=15,
        style='italic', fontweight='bold', transform=ax.transAxes)
ax.annotate("", xy=(0.5, 0.70), xytext=(0.5, 0.80), xycoords='axes fraction',
            arrowprops=dict(arrowstyle="-|>", lw=3, color='#333'))
ax.text(0.55, 0.745, "CountVectorizer", fontsize=11, style='italic', color='#666',
        transform=ax.transAxes)

cols = ["win", "free", "prize", "now", "lunch", "meeting", "...", "hello"]
vals = ["1", "1", "1", "1", "0", "0", "...", "0"]
n = len(cols); w = 0.10; x0 = 0.5 - n*w/2
for i, (c, v) in enumerate(zip(cols, vals)):
    on = v == "1"
    ax.add_patch(mp.Rectangle((x0+i*w, 0.45), w, 0.13, facecolor='#FFF3C4' if on else 'white',
                              edgecolor='black', lw=1.5, transform=ax.transAxes))
    ax.add_patch(mp.Rectangle((x0+i*w, 0.58), w, 0.09, facecolor='#E8E8E8',
                              edgecolor='black', lw=1.5, transform=ax.transAxes))
    ax.text(x0+i*w+w/2, 0.625, c, ha='center', va='center', fontsize=10,
            fontweight='bold', transform=ax.transAxes)
    ax.text(x0+i*w+w/2, 0.515, v, ha='center', va='center', fontsize=15,
            fontweight='bold', color=RED if on else '#BBB', transform=ax.transAxes)

ax.text(0.5, 0.31, "Every word in the vocabulary becomes a COLUMN.\nThe value is how many times it appears in the message.",
        ha='center', fontsize=13, transform=ax.transAxes)
ax.text(0.5, 0.14, "Word order is LOST -- it's a 'bag', not a sentence.\nAnd it still works remarkably well.",
        ha='center', fontsize=12.5, style='italic', color='#555', transform=ax.transAxes)
plt.savefig(os.path.join(OUT, "nb_bag_of_words.png"), dpi=200, bbox_inches='tight')
plt.savefig(os.path.join(OUT, "nb_bag_of_words.svg"), bbox_inches='tight')
print("3 ok")

1 ok
2 ok
3 ok


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("sms_spam.csv")
print("Shape:", df.shape)
df.head()

Shape: (5572, 2)


,message,spam
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


In [5]:
print(df["spam"].value_counts())
print(f"\nSpam rate: {df['spam'].mean():.1%}")

spam
0    4825
1     747
Name: count, dtype: int64

Spam rate: 13.4%


In [6]:
print("--- HAM (legitimate) ---")
for msg in df[df.spam == 0]["message"].head(2):
    print(" *", msg[:80])

print("\n--- SPAM ---")
for msg in df[df.spam == 1]["message"].head(2):
    print(" *", msg[:80])

--- HAM (legitimate) ---
 * Go until jurong point, crazy.. Available only in bugis n great world la e buffet
 * Ok lar... Joking wif u oni...

--- SPAM ---
 * Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 8
 * FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some


In [7]:
from sklearn.feature_extraction.text import CountVectorizer

# tiny demo to see what it does
demo = ["win a free prize now", "are we having lunch now"]
demo_vec = CountVectorizer()
demo_matrix = demo_vec.fit_transform(demo)

pd.DataFrame(
    demo_matrix.toarray(),
    columns=demo_vec.get_feature_names_out(),
    index=["message 1", "message 2"]
)

# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html


,are,free,having,lunch,now,prize,we,win
message 1,0,1,0,0,1,1,0,1
message 2,1,0,1,1,1,0,1,0


In [8]:
from sklearn.model_selection import train_test_split

X = df["message"]
y = df["spam"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", len(X_train), "| Test:", len(X_test))

Train: 4457 | Test: 1115


In [9]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
import time

model = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", MultinomialNB()),
])

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

print(f"Trained in {elapsed:.3f} seconds")

Trained in 0.093 seconds


In [10]:

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

predictions = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2%}\n")
print(classification_report(y_test, predictions,
                            target_names=["ham", "spam"], digits=3))

Accuracy: 98.74%

              precision    recall  f1-score   support

         ham      0.988     0.998     0.993       966
        spam      0.986     0.919     0.951       149

    accuracy                          0.987      1115
   macro avg      0.987     0.959     0.972      1115
weighted avg      0.987     0.987     0.987      1115



In [16]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, predictions),
    display_labels=["ham", "spam"]
).plot(ax=ax, cmap="Blues")
plt.title("Spam detector — confusion matrix")
plt.show()

In [15]:
nb = model.named_steps["classifier"]
vec = model.named_steps["vectorizer"]
words = np.array(vec.get_feature_names_out())

# log-ratio: how much more likely is this word in spam than in ham?
log_ratio = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

top_spam = words[np.argsort(log_ratio)[-12:][::-1]]
top_ham = words[np.argsort(log_ratio)[:8]]

print("Most SPAM-ish words:", ", ".join(top_spam))
print("\nMost HAM-ish words: ", ", ".join(top_ham))

Most SPAM-ish words: claim, prize, 150p, tone, 18, cs, 500, guaranteed, 1000, uk, landline, 150ppm

Most HAM-ish words:  gt, lt, he, da, lor, she, later, amp


In [17]:
new_messages = [
    "WINNER!! You have won a FREE prize. Claim now, call 09061701461",
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! Your account will be suspended. Click here to verify",
]

probs = model.predict_proba(new_messages)

for msg, p in zip(new_messages, probs):
    verdict = "SPAM" if p[1] > 0.5 else "ham"
    print(f"[{verdict:4s}] spam probability: {p[1]:6.1%}  |  {msg[:50]}...")

[SPAM] spam probability: 100.0%  |  WINNER!! You have won a FREE prize. Claim now, cal...
[ham ] spam probability:   0.0%  |  Hey, are we still meeting for lunch tomorrow?...
[SPAM] spam probability:  97.1%  |  URGENT! Your account will be suspended. Click here...


In [18]:

from sklearn.neighbors import KNeighborsClassifier

knn = Pipeline([
    ("vectorizer", CountVectorizer()),
    ("classifier", KNeighborsClassifier()),
])

t0 = time.time(); knn.fit(X_train, y_train); knn_fit = time.time() - t0
t0 = time.time(); knn_pred = knn.predict(X_test); knn_predict = time.time() - t0

t0 = time.time(); nb_pred = model.predict(X_test); nb_predict = time.time() - t0

print(f"Naive Bayes -> accuracy {accuracy_score(y_test, nb_pred):.2%} | predict: {nb_predict:.3f}s")
print(f"KNN         -> accuracy {accuracy_score(y_test, knn_pred):.2%} | predict: {knn_predict:.3f}s")
print(f"\nKNN is ~{knn_predict/nb_predict:.0f}x slower to predict.")

Naive Bayes -> accuracy 98.74% | predict: 0.033s
KNN         -> accuracy 91.84% | predict: 0.276s

KNN is ~8x slower to predict.
